# CENG 786 - ROBOT MOTION PLANNING AND CONTROL
## Assignment #1: Implementing Tangent Bug Algorithm

##### Çağdaş Güven
##### MIDDLE EAST TECHNICAL UNIVERSITY - Robotics 
##### Instructor: Prof. Uluç Saranlı

### Introduction

The **Tangent Bug Algorithm** is a path planning method widely used in robotics for navigating unknown environments while avoiding obstacles. Unlike global planners that rely on full knowledge of the environment, the Tangent Bug Algorithm operates with local sensing, making real-time decisions based on the robot's immediate surroundings. By combining goal-directed movement with obstacle avoidance through boundary-following and tangent-based navigation, the algorithm allows the robot to reach a target efficiently, even in dynamic or partially known environments. This report details the implementation of the Tangent Bug Algorithm, exploring its functionality, performance, and application in real-world scenarios.

Here are the key formulas related to the A* algorithm and the Tangent Boundary Following Bug approach:

### 1. **A* Algorithm Formulas**
The A* algorithm is based on evaluating a cost function $\( f(x) \) $for each node$ \( x \):$

$
f(x) = g(x) + h(x)
$

where:
- $ g(x) $ is the actual cost from the start node to node $x $.
- $h(x) $ is the heuristic estimate of the cost from node $ x $ to the goal.

#### **Cost Calculation**
- **Movement Cost $( g(x) )$**:
  $
  g(x) = g(\text{parent of } x) + \text{cost to move from parent to } x
  $
  The cost can be calculated based on the distance between adjacent nodes, e.g., Euclidean distance:
  $
  \text{cost} = \sqrt{(x_2 - x_1)^2 + (y_2 - y_1)^2}
  $

- **Heuristic Cost $( h(x) )$**:
  Typically, the Euclidean distance from the current node $ x $ to the goal $ q_{\text{goal}} :$
  $
  h(x) = \sqrt{(x - q_{\text{goal}}(1))^2 + (y - q_{\text{goal}}(2))^2}
  $

### 2. **Tangent Boundary Following Formulas**
When encountering obstacles, the robot switches to tangent boundary following. The key formulas are related to maintaining a safe distance and navigating tangentially:

#### **Heuristic for Choosing Boundary Points**
During boundary following, the robot evaluates possible boundary points $ n $ based on a heuristic:
$
\text{Heuristic: } h(x, n) = d(x, n) + d(n, q_{\text{goal}})
$
where:
- $ d(x, n) $ is the distance between the current position $x $ and the boundary point $ n $.
- $ d(n, q_{\text{goal}}) $ is the distance between the boundary point $ n  $ and the goal position.

#### **Maintaining Tangent Distance**
- **Safe Distance**:
  The robot maintains a distance between a minimum $( d_{\text{min}} ) $ and maximum $ (d_{max})  $ threshold:
  $
  d_{\text{min}} \leq \text{distance to obstacle} \leq d_{\text{max}}
  $
- **Perpendicular Adjustment**:
  When the robot gets too close or too far from the obstacle, it adjusts its path:
  $
  \text{Adjustment} = d_{\text{desired}} - \text{distance to obstacle}
  $
where $ d_{\text{desired}}  $ is within the range $ [d_{\text{min}}, d_{\text{max}}]$.

These formulas drive the core logic of our integrated A* and Tangent Bug approach, allowing the robot to plan paths efficiently while avoiding obstacles and maintaining smooth navigation along boundaries when necessary.

### Related Work

### Algorithm Description

The **Tangent Bug Algorithm** is a reactive path planning algorithm designed for robots navigating in environments with obstacles using local sensing. It combines two primary behaviors: 

1. **Goal Seeking**: The robot moves directly toward the target as long as no obstacles are detected in its path.
   
2. **Obstacle Avoidance**: When the robot encounters an obstacle, it computes tangent lines to the obstacle's boundary and selects the best direction to follow. It either follows the boundary or switches back to goal-seeking when it finds a clear path.

By switching between these behaviors, the robot effectively avoids obstacles while continuously trying to minimize the distance to the goal. The algorithm does not require a full map of the environment and relies on local sensor data, making it suitable for unknown or dynamic environments.

### Implementation Details

Here's a breakdown of the implementation details, explaining how the algorithm switches between A* path planning and tangent boundary following, how it detects obstacles, and why implementing tangent following can be challenging.

### 1. **Switching Between A* Planning and Tangent Boundary Following**

#### A* Path Planning:
- **Primary Behavior**: The algorithm starts by planning a path using A* search. This mode tries to find a direct path from the start (`qstart`) to the goal (`qgoal`).
- **Path Expansion**: A* explores potential paths by expanding nodes (points in the arena) and calculating the cost (`g_score`), which is the movement cost from the start, and the estimated cost to the goal (`f_score`), which is `g_score + heuristic`.

#### Detecting When to Switch:
- **Obstacle Detection**: During path expansion, the algorithm uses a function (`is_obstacle`) that checks if a move in a particular direction would result in a collision. If an obstacle is detected, the algorithm switches from A* to tangent boundary following.
- **When Tangent Following Starts**: The algorithm enters the tangent boundary-following mode when:
  - The sensor data indicates there is an obstacle blocking the direct path to the goal.
  - The robot is too close to the obstacle and requires careful maneuvering to avoid a collision.

### 2. **How the Algorithm Notices the Obstacle**

#### Sensor-Based Detection:
- **Sensor Mechanism**: The robot is equipped with a 360-degree sensor (`read_sensor`) that measures the distance to the nearest obstacle at any given angle. 
- **Continuous Monitoring**: As the robot navigates, it continuously checks the sensor data in the direction of its intended movement. If the distance is less than a threshold (indicating the presence of an obstacle), it switches to boundary following.
- **Range and Discontinuities**: When the robot sweeps the sensor, it can detect "discontinuities" or significant changes in distance measurements. These indicate potential edges or gaps in obstacles, which can be useful for tangent following.

### 3. **How the Algorithm Avoids Obstacles**

#### Tangent Boundary Following:
- **Maintaining a Safe Distance**: Once an obstacle is detected, the robot needs to navigate around it without colliding. The tangent boundary-following behavior helps achieve this by:
  - **Adjusting Position**: When an obstacle is detected, the robot adjusts its position to maintain a distance within a specified range (`min_distance` to `max_distance`). This ensures the robot does not get too close to or too far from the obstacle while following it.
  - **Finding and Following the Boundary**: The algorithm identifies the boundary edges using sensor readings. It then moves tangentially along the obstacle’s boundary, maintaining a safe distance and adjusting the movement if it gets too close or far.

#### Resuming A* Path Planning:
- **Switching Back**: The robot continuously checks if it can see a clear path to the goal while following the boundary. When it detects that the path to the goal is unobstructed, it switches back to A* planning and continues towards the goal.
- **Heuristic and Edge Calculation**: The robot uses a heuristic to determine the shortest path, even while avoiding obstacles. It chooses edges that minimize the total cost (current distance + estimated distance to the goal), helping it make intelligent choices when navigating around obstacles.

### 4. **Why Tangent Following Was Hard to Implement**

#### Challenges with Tangent Following:
1. **Continuous Adjustments**:
   - Unlike simple obstacle avoidance where the robot can stop or turn around immediately, tangent following requires the robot to continuously adjust its path to remain at a safe distance from the boundary. This involves complex calculations to ensure the robot does not oscillate or veer off course.
   - Implementing precise adjustments based on continuous sensor feedback is challenging because any slight delay or incorrect calculation can lead to erratic behavior or collisions.

2. **Determining When to Switch Modes**:
   - One of the biggest challenges is deciding when to switch from tangent following back to A* planning. The algorithm needs to frequently check if the goal is visible and reachable without obstacles, which involves continuously reading sensor data and recalculating paths.
   - Improper switching can cause the robot to get "stuck" in a loop, either constantly switching between modes or failing to leave tangent following even when the path to the goal is clear.

3. **Boundary Edge Detection**:
   - Detecting edges accurately can be difficult, especially when there are complex-shaped obstacles or overlapping obstacles. The robot needs to understand which parts of the sensor readings correspond to actual boundaries and which are just gaps or noise.
   - Misidentification of edges could lead the robot to take inefficient or incorrect paths, making tangent following less effective.

4. **Smooth Path Execution**:
   - Following a smooth, tangential path requires the robot to calculate angles, distances, and adjustments in real-time. Any abrupt changes in direction can lead to jerky movements, which can be particularly problematic when navigating close to obstacles.
   - Implementing smooth path execution involves balancing between avoiding obstacles and maintaining forward progress toward the goal, which is a non-trivial problem in robotics.

### Results and Evaluation

Our algorithm integrates A* path planning with optional tangent boundary following, allowing the robot to navigate complex environments effectively:
1. **A* Path Planning** ensures the robot follows a globally optimal path to the goal. The algorithm expands nodes based on cost (`g_score`) and estimates distance to the goal (`f_score`).
2. **Tangent Boundary Following** activates when obstacles block the path. The robot uses sensor data to detect and navigate around obstacles, maintaining a safe distance. This mode terminates once a clear path to the goal is visible, allowing the robot to switch back to A*.

Compared to the **TangentBug** algorithm by Kamon et al., our method uses a similar local-global approach but relies on A* for efficient global pathfinding, with tangent following providing localized obstacle navigation【168†source】. Similar to the **JAUS compliant mobile robot control**, our tangent following can be adapted for real-world conditions by accounting for robot dimensions, ensuring feasible boundary-following paths【167†source】.



### Discussion

Implementing tangent following posed challenges:
1. **Dynamic Adjustments**: Continuously adjusting the path based on real-time sensor data can lead to abrupt movements. Maintaining a smooth distance from obstacles requires precise control, particularly with irregular surfaces.
2. **Seamless Mode Switching**: Ensuring smooth transitions between A* and tangent following was critical. The robot must detect when it can resume direct movement to the goal without getting stuck, echoing the global convergence condition seen in **TangentBug**【168†source】.
3. **Practicality of Tangent Paths**: Real-world implementations, such as those in **LTG-based approaches**, show the need for adapting local graphs to handle irregular obstacles. Our approach uses local sensor data dynamically, but smooth integration into global pathfinding can still be complex.

Overall, combining A* with optional tangent following provides a balanced strategy between global efficiency and local adaptability, enabling better performance across diverse environments.

### Conclusion

The integration of A* planning with optional tangent boundary following allows for a more intelligent navigation system. A* provides an overall path to the goal, while tangent following ensures that the robot can handle obstacles gracefully when they block its path. Despite the challenges, combining these approaches gives the robot flexibility in different environments, allowing it to adapt to both open spaces and more cluttered, obstacle-heavy areas.

### References

add given references (maybe add 1 or 2 more )
add textbook as reference

add visuals for testings and for examples maybe from internet

### Appendix